<a href="https://colab.research.google.com/github/Minhaj401/nlp/blob/main/simple_rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
from keras.datasets import imdb
from keras.models import Sequential
from keras.layers import Embedding, SimpleRNN, Dense
from keras.preprocessing.sequence import pad_sequences

In [44]:

 (X_train,y_train),(X_test,y_test) = imdb.load_data()

In [45]:
X_train = pad_sequences(X_train,padding='post',maxlen=50)
X_test = pad_sequences(X_test,padding='post',maxlen=50)

In [46]:
y_test

array([0, 1, 1, ..., 0, 0, 0])

In [47]:
model = Sequential()
model.add(Embedding(100000, 2, input_length=50))
model.add(SimpleRNN(32,return_sequences=False))
model.add(Dense(1, activation='sigmoid'))

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [48]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])
history = model.fit(X_train, y_train,epochs=5,validation_data=(X_test,y_test))

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 18s 20ms/step - acc: 0.6196 - loss: 0.6274 - val_acc: 0.7906 - val_loss: 0.4553
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - acc: 0.8332 - loss: 0.3812 - val_acc: 0.8041 - val_loss: 0.4268
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - acc: 0.8978 - loss: 0.2622 - val_acc: 0.7990 - val_loss: 0.4411
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - acc: 0.9302 - loss: 0.1911 - val_acc: 0.7946 - val_loss: 0.5058
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 19ms/step - acc: 0.9515 - loss: 0.1419 - val_acc: 0.7930 - val_loss: 0.5512


### Test the model with your own input

Now you can enter a sentence or phrase, and the model will predict its sentiment.

In [49]:
# Get user input
user_input_text = input("Enter a sentence for sentiment prediction: ")
print(f"You entered: {user_input_text}")

Enter a sentence for sentiment prediction: bad movie
You entered: bad movie


In [50]:
# Preprocess the user input
# The IMDB dataset does not use a Tokenizer object like the one from keras.preprocessing.text.
# Instead, it provides a word_index mapping.
word_index = imdb.get_word_index()

# Prepare a mapping for unknown words and special tokens
# The indices in the IMDB dataset are typically offset by 3:
# 0: padding, 1: start of sequence, 2: unknown, 3: unused
indexed_review = []
for word in user_input_text.lower().split():
    # Get the index for the word, default to 0 if not found (which becomes 3 after offset)
    # Add 3 to the word index to match the format used by imdb.load_data()
    idx = word_index.get(word, 0) + 3
    # Ensure index is within the Embedding layer's vocabulary size (10000)
    if idx >= 10000:
        indexed_review.append(2) # Map to unknown (index 2) if outside vocabulary size
    else:
        indexed_review.append(idx)

# Wrap in a list for batch processing
user_sequence_list = [indexed_review]

# Pad the sequence to match the input_length of the model (maxlen=50)
# Make sure to use the same padding style ('post')
user_padded_sequence = pad_sequences(user_sequence_list, padding='post', maxlen=50)

print("Preprocessed sequence:")
print(user_padded_sequence)

Preprocessed sequence:
[[78 20  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0]]


In [51]:
# Make a prediction
prediction = model.predict(user_padded_sequence)

# The output is a probability. For binary classification, > 0.5 can be considered positive.
if prediction[0][0] > 0.5:
    sentiment = 'Positive'
else:
    sentiment = 'Negative'

print(f"\nPrediction probability: {prediction[0][0]:.4f}")
print(f"Predicted sentiment: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step

Prediction probability: 0.0184
Predicted sentiment: Negative
